In [ ]:
# Once the package is installed, we can access the GlioTrace by running
from gliotrace.gliotrace_class import GlioTrace

In [ ]:
import pandas as pd
from glob import glob

# ----------------------------------------------------------------------
# Run the pipeline with the provided example data
# ----------------------------------------------------------------------

stackfile = glob("Example_data/**/*.npz", recursive = True)

metadata = pd.DataFrame(
    {
        "experiment_id": [329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339],
        "delta_t":[1.45, 1.45, 1.45, 1.45, 1.45, 1.45, 1.35, 1.35, 1.35, 1.35, 1.35],
        "set": [67, 67, 67, 67, 67, 67, 68, 68, 68, 68, 68],
        "perturbation": ["control", "control", "dasatinib", "dasatinib", "thapsigargin", "thapsigargin", "control", "control", "thapsigargin", "thapsigargin", "dasatinib"],
        "dose": [0, 0, 32, 32, 4, 4, 0, 0, 4, 4, 32],
        "unit": ["uM", "uM", "uM", "uM", "uM", "uM", "uM", "uM", "uM", "uM", "uM"]
    }
)

fcols = ["vascular_distance"] # <---- for the logistic model in the hmm, 
                              #       can be changed after tracking, see 
                              #       below in the gliobj.loo_hmm() section
            

gliobj_dasatinib = GlioTrace(stackfile=stackfile, 
                             metadata=metadata,
                             fcols=fcols,
                             control = "control",
                             treatment = ["dasatinib", "32"])

gliobj_thapsigargin = GlioTrace(stackfile=stackfile, 
                             metadata=metadata,
                             fcols=fcols,
                             control = "control",
                             treatment = ["thapsigargin", "4"])

In [ ]:
# Run cell detection and tracking (estimated run time on a 16Gb M2 mac: 3.5 hrs)
gliobj_dasatinib.run_tracking(save_point="Saved_runs/run1")

gliobj_thapsigargin.run_tracking(save_point="Saved_runs/run2")

In [ ]:
# Load runs
gliobj_dasatinib = GlioTrace.load_run("Saved_runs/run1") 
gliobj_thapsigargin = GlioTrace.load_run("Saved_runs/run2") 

In [ ]:
# Run LOO_HMM runs (estimated run time on a 16Gb M2 mac: 20 mins)
results_dasat = gliobj_dasatinib.loo_hmm(
    fcols=["vascular_distance", "polarization", "tme_label", "is_treatment"],
    save_path="runs_dasat/loo/",
)

results_thaps = gliobj_thapsigargin.loo_hmm(
    fcols=["vascular_distance", "polarization", "tme_label", "is_treatment"],
    save_path="runs_thaps/loo/",
)

In [ ]:
# Print output
results_dasat
results_thaps